# UTS Pembelajaran Mesin 2026 — Prediksi Diabetes
**Algoritma:** Decision Tree (Classifier) & Regresi Linear (Linear Regression)  
**Dataset:** `02_diabetes.csv` (100.000 baris, 9 kolom, target = `diabetes`)  

## Langkah 1 — Import library & konfigurasi

In [ ]:
#202431100_SaidAqielSyafaraz
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, mean_absolute_error,
                             mean_squared_error, r2_score)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

RANDOM_STATE = 42         # agar hasil dapat direproduksi

print("Library berhasil diimport.")

## Langkah 2 — Memuat dataset

In [ ]:
#202431100_SaidAqielSyafaraz
# Colab akan menampilkan tombol "Choose Files" -> pilih 02_diabetes.csv
try:
    from google.colab import files
    uploaded = files.upload()
    nama_file = list(uploaded.keys())[0]
except ImportError:                      # jika dijalankan di komputer lokal
    nama_file = "02_diabetes.csv"

df = pd.read_csv(nama_file)
print("Dataset dimuat dari:", nama_file)
print("Ukuran dataset (baris, kolom):", df.shape)

## Langkah 3 — Memahami dataset (Data Understanding)

In [ ]:
#202431100_SaidAqielSyafaraz
print("=== 5 BARIS PERTAMA ===")
display(df.head())
print("\n=== INFO DATASET ===")
df.info()
print("\n=== STATISTIK DESKRIPTIF ===")
display(df.describe().T)

## Langkah 4 — EDA: kualitas data (missing value, duplikat, kategori)

In [ ]:
#202431100_SaidAqielSyafaraz
print("=== MISSING VALUE PER KOLOM ===")
print(df.isnull().sum())
print("\nJumlah baris duplikat :", df.duplicated().sum())
print("\n=== NILAI UNIK KOLOM KATEGORIKAL ===")
print("gender          :", df["gender"].value_counts().to_dict())
print("smoking_history :", df["smoking_history"].value_counts().to_dict())

## Langkah 5 — EDA: distribusi variabel target

In [ ]:
#202431100_SaidAqielSyafaraz
jumlah = df["diabetes"].value_counts()
persen = df["diabetes"].value_counts(normalize=True) * 100
print(pd.DataFrame({"jumlah": jumlah, "persen (%)": persen.round(2)}))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="diabetes", data=df, palette=["#4C9F70", "#D1495B"], ax=ax)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
ax.set_title("Distribusi Kelas Target (0 = Tidak Diabetes, 1 = Diabetes)")
plt.tight_layout(); plt.show()

## Langkah 6 — EDA: distribusi fitur numerik & deteksi outlier

In [ ]:
#202431100_SaidAqielSyafaraz
num_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[0, i], color="#2E86AB")
    axes[0, i].set_title(f"Histogram {col}")
    sns.boxplot(x=df[col], ax=axes[1, i], color="#F6AE2D")
    axes[1, i].set_title(f"Boxplot {col}")
plt.tight_layout(); plt.show()

# Hitung jumlah outlier dengan aturan IQR
print("=== JUMLAH OUTLIER (aturan IQR) ===")
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f"{col:22s}: {n_out} data ({n_out/len(df)*100:.2f}%)")

## Langkah 7 — EDA: hubungan fitur dengan target

In [ ]:
#202431100_SaidAqielSyafaraz
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), ["gender", "smoking_history", "hypertension", "heart_disease"]):
    rate = df.groupby(col)["diabetes"].mean() * 100
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax, palette="viridis")
    ax.set_title(f"Persentase penderita diabetes berdasarkan {col}")
    ax.set_ylabel("% diabetes"); ax.set_xlabel(col)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## Langkah 8 — EDA: matriks korelasi

In [ ]:
#202431100_SaidAqielSyafaraz
korelasi = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(korelasi, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriks Korelasi Fitur Numerik")
plt.tight_layout(); plt.show()

print("Korelasi fitur terhadap target 'diabetes':")
print(korelasi["diabetes"].drop("diabetes").sort_values(ascending=False).round(3))

## Langkah 9 — Preprocessing: pembersihan & encoding

In [ ]:
#202431100_SaidAqielSyafaraz
df_clean = df.copy()

# 1) Hapus baris duplikat
sebelum = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Duplikat dihapus      : {sebelum - len(df_clean)} baris")

# 2) Hapus kategori 'Other' pada gender (hanya 18 data, terlalu sedikit)
sebelum = len(df_clean)
df_clean = df_clean[df_clean["gender"] != "Other"]
print(f"Gender 'Other' dihapus: {sebelum - len(df_clean)} baris")

# 3) Encoding gender -> biner (Female = 0, Male = 1)
df_clean["gender"] = df_clean["gender"].map({"Female": 0, "Male": 1})

# 4) One-Hot Encoding smoking_history (drop_first menghindari dummy variable trap)
df_clean = pd.get_dummies(df_clean, columns=["smoking_history"], drop_first=True, dtype=int)

df_clean = df_clean.reset_index(drop=True)
print("\nUkuran data setelah preprocessing:", df_clean.shape)
display(df_clean.head())
print("Missing value tersisa:", df_clean.isnull().sum().sum())

## Langkah 10 — Pembagian data latih & uji (80 : 20, stratified)

In [ ]:
#202431100_SaidAqielSyafaraz
X = df_clean.drop(columns="diabetes")
y = df_clean["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Proporsi diabetes pada data latih :", round(y_train.mean()*100, 2), "%")
print("Proporsi diabetes pada data uji   :", round(y_test.mean()*100, 2), "%")

## Langkah 11 — Normalisasi fitur (StandardScaler) untuk Regresi Linear

In [ ]:
#202431100_SaidAqielSyafaraz
# Decision Tree tidak sensitif terhadap skala -> memakai data asli.
# Regresi Linear dibantu normalisasi -> scaler HANYA di-fit pada data latih (mencegah data leakage).
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns, index=X_test.index)

print("Rata-rata fitur latih setelah scaling (mendekati 0):")
print(X_train_sc.mean().round(3))

## Langkah 12 — Decision Tree: model dasar (default)

In [ ]:
#202431100_SaidAqielSyafaraz
dt_base = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_base.fit(X_train, y_train)

acc_train = accuracy_score(y_train, dt_base.predict(X_train))
acc_test  = accuracy_score(y_test,  dt_base.predict(X_test))
print(f"Kedalaman pohon      : {dt_base.get_depth()}")
print(f"Jumlah daun          : {dt_base.get_n_leaves()}")
print(f"Akurasi data latih   : {acc_train:.4f}")
print(f"Akurasi data uji     : {acc_test:.4f}")
print("-> Selisih latih-uji yang besar menandakan overfitting.")

## Langkah 13 — Decision Tree: tuning hyperparameter (GridSearchCV)

In [ ]:
#202431100_SaidAqielSyafaraz
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "min_samples_leaf": [1, 10, 50],
}
grid = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                    param_grid, cv=5, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

dt_best = grid.best_estimator_
print("Parameter terbaik :", grid.best_params_)
print(f"F1 rata-rata CV   : {grid.best_score_:.4f}")
print(f"Akurasi data latih: {accuracy_score(y_train, dt_best.predict(X_train)):.4f}")
print(f"Akurasi data uji  : {accuracy_score(y_test,  dt_best.predict(X_test)):.4f}")

## Langkah 14 — Decision Tree: evaluasi lengkap

In [ ]:
#202431100_SaidAqielSyafaraz
y_pred_dt  = dt_best.predict(X_test)
y_prob_dt  = dt_best.predict_proba(X_test)[:, 1]

hasil_dt = {
    "Akurasi":   accuracy_score(y_test, y_pred_dt),
    "Presisi":   precision_score(y_test, y_pred_dt),
    "Recall":    recall_score(y_test, y_pred_dt),
    "F1-Score":  f1_score(y_test, y_pred_dt),
    "ROC-AUC":   roc_auc_score(y_test, y_prob_dt),
}
for k, v in hasil_dt.items():
    print(f"{k:10s}: {v:.4f}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_dt, target_names=["Tidak Diabetes", "Diabetes"]))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_dt),
                       display_labels=["Tidak", "Diabetes"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — Decision Tree")
plt.tight_layout(); plt.show()

## Langkah 15 — Decision Tree: visualisasi pohon & feature importance

In [ ]:
#202431100_SaidAqielSyafaraz
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_best, feature_names=X.columns, class_names=["Tidak", "Diabetes"],
          filled=True, rounded=True, max_depth=3, fontsize=9, ax=ax)
ax.set_title("Visualisasi Decision Tree (ditampilkan sampai kedalaman 3)")
plt.tight_layout(); plt.show()

imp = pd.Series(dt_best.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind="barh", color="#2E86AB", ax=ax)
ax.set_title("Feature Importance — Decision Tree")
plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(4))

## Langkah 16 — Regresi Linear: pelatihan model

In [ ]:
#202431100_SaidAqielSyafaraz
lin_reg = LinearRegression()
lin_reg.fit(X_train_sc, y_train)

koef = pd.Series(lin_reg.coef_, index=X.columns).sort_values(ascending=False)
print("Intercept :", round(lin_reg.intercept_, 4))
print("\nKoefisien regresi (fitur sudah di-scaling):")
print(koef.round(4))

## Langkah 17 — Regresi Linear: evaluasi (metrik regresi + akurasi klasifikasi)

In [ ]:
#202431100_SaidAqielSyafaraz
# Regresi linear menghasilkan nilai kontinu. Target bersifat biner (0/1),
# sehingga hasil prediksi dikonversi: >= 0.5 -> kelas 1 (diabetes), < 0.5 -> kelas 0.
y_cont_lr = lin_reg.predict(X_test_sc)
y_pred_lr = (y_cont_lr >= 0.5).astype(int)

print("=== METRIK REGRESI ===")
print(f"MAE  : {mean_absolute_error(y_test, y_cont_lr):.4f}")
print(f"MSE  : {mean_squared_error(y_test, y_cont_lr):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_cont_lr)):.4f}")
print(f"R2   : {r2_score(y_test, y_cont_lr):.4f}")

hasil_lr = {
    "Akurasi":   accuracy_score(y_test, y_pred_lr),
    "Presisi":   precision_score(y_test, y_pred_lr, zero_division=0),
    "Recall":    recall_score(y_test, y_pred_lr),
    "F1-Score":  f1_score(y_test, y_pred_lr),
    "ROC-AUC":   roc_auc_score(y_test, y_cont_lr),
}
print("\n=== METRIK KLASIFIKASI (threshold 0.5) ===")
for k, v in hasil_lr.items():
    print(f"{k:10s}: {v:.4f}")
print("\n", classification_report(y_test, y_pred_lr, target_names=["Tidak Diabetes", "Diabetes"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr),
                       display_labels=["Tidak", "Diabetes"]).plot(ax=axes[0], cmap="Oranges", colorbar=False)
axes[0].set_title("Confusion Matrix — Regresi Linear")
sns.histplot(x=y_cont_lr, hue=y_test.values, bins=50, ax=axes[1], palette=["#4C9F70", "#D1495B"])
axes[1].axvline(0.5, color="black", linestyle="--", label="threshold 0.5")
axes[1].set_title("Distribusi nilai prediksi Regresi Linear"); axes[1].legend()
plt.tight_layout(); plt.show()

## Langkah 18 — Perbandingan Decision Tree vs Regresi Linear

In [ ]:
#202431100_SaidAqielSyafaraz
tabel = pd.DataFrame({"Decision Tree": hasil_dt, "Regresi Linear": hasil_lr}).round(4)
display(tabel)

ax = tabel.drop(index="ROC-AUC").plot(kind="bar", figsize=(9, 5), color=["#2E86AB", "#F6AE2D"], rot=0)
ax.set_ylim(0, 1.05); ax.set_title("Perbandingan Kinerja Model pada Data Uji")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.show()

terbaik = tabel.loc["Akurasi"].idxmax()
print(f"Model dengan akurasi tertinggi: {terbaik} ({tabel.loc['Akurasi'].max():.4f})")

## Langkah 19 — Uji prediksi pada contoh pasien baru

In [ ]:
#202431100_SaidAqielSyafaraz
pasien = pd.DataFrame([{
    "gender": 1, "age": 55, "hypertension": 1, "heart_disease": 0, "bmi": 31.5,
    "HbA1c_level": 7.0, "blood_glucose_level": 200,
    "smoking_history_current": 0, "smoking_history_ever": 0, "smoking_history_former": 1,
    "smoking_history_never": 0, "smoking_history_not current": 0,
}])[X.columns]          # urutan kolom disamakan dengan data latih

p_dt = dt_best.predict(pasien)[0]
p_lr = lin_reg.predict(scaler.transform(pasien))[0]
print("Data pasien:"); display(pasien.T.rename(columns={0: "nilai"}))
print(f"Prediksi Decision Tree : {'DIABETES' if p_dt == 1 else 'TIDAK DIABETES'}")
print(f"Prediksi Regresi Linear: nilai {p_lr:.3f} -> {'DIABETES' if p_lr >= 0.5 else 'TIDAK DIABETES'}")

## Langkah 20 — Kesimpulan otomatis

In [ ]:
#202431100_SaidAqielSyafaraz
print("RINGKASAN HASIL UTS")
print("-" * 55)
print(f"Jumlah data setelah preprocessing : {len(df_clean)}")
print(f"Akurasi Decision Tree             : {hasil_dt['Akurasi']*100:.2f}%")
print(f"Akurasi Regresi Linear (thr 0.5)  : {hasil_lr['Akurasi']*100:.2f}%")
print(f"Recall   Decision Tree / Reg.Lin. : {hasil_dt['Recall']:.3f} / {hasil_lr['Recall']:.3f}")
print(f"F1-Score Decision Tree / Reg.Lin. : {hasil_dt['F1-Score']:.3f} / {hasil_lr['F1-Score']:.3f}")
print("-" * 55)
print("Model terbaik berdasarkan akurasi :", terbaik)